# Mimi-VC: Full Experimental Evaluation

This notebook runs all experiments for the paper:
**"Mimi-VC: Voice Conversion via Disentangled Neural Audio Codec Token Manipulation"**

**Experiments:**
- Table I: Objective evaluation (SECS, CER, MCD, PESQ, RTF)
- Table II: Subjective evaluation setup (MOS/SMOS audio export)
- Table III: Computational cost
- Table IV: Codebook ablation study
- Retrieval parameter sweep
- Transformer design ablation

**Dataset:** LibriSpeech test-clean (40 speakers, ~5.4 hours)

> Runtime: **GPU T4** required. Full run ~45 min.

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers torch torchaudio datasets faiss-cpu \
    librosa matplotlib pandas resemblyzer pesq jiwer openai-whisper

In [ ]:
# ============================================================
# 2. IMPORTS & CONFIGURATION
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import whisper
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import json
import warnings
from collections import defaultdict
from pathlib import Path
from datasets import load_dataset
from transformers import MimiModel, AutoFeatureExtractor
from resemblyzer import VoiceEncoder, preprocess_wav
from pesq import pesq as pesq_fn
from jiwer import cer as compute_cer
from IPython.display import Audio as PlayAudio, display

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# === Experiment Config ===
N_SOURCE_SPEAKERS = 5       # number of source speakers
N_TARGET_SPEAKERS = 5       # number of target speakers
N_EVAL_UTTS = 5             # utterances per source for eval
TRANSFORMER_STEPS = 500     # training steps for translator
TRANSFORMER_BATCH = 8
TRANSFORMER_LR = 3e-4
SEQ_LEN = 64                # frames per training sequence
RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Config loaded.')

In [ ]:
# ============================================================
# 3. LOAD MODELS
# ============================================================

# --- Mimi Codec ---
print('Loading Mimi codec...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')
SR = feat_ext.sampling_rate           # 24000
N_CB = mimi.config.num_quantizers     # 8
CB_SIZE = mimi.config.codebook_size   # 2048
FRAME_RATE = mimi.config.frame_rate   # 12.5
print(f'  SR={SR}, Codebooks={N_CB}x{CB_SIZE}, Frame rate={FRAME_RATE} Hz')

# --- Speaker Encoder (Resemblyzer) ---
print('Loading speaker encoder...')
spk_encoder = VoiceEncoder(device=device)

# --- ASR (Whisper) ---
print('Loading Whisper ASR...')
whisper_model = whisper.load_model('base', device=device)

print('All models loaded.')

In [ ]:
# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def encode_audio(wav_np):
    """Encode numpy audio to Mimi tokens [1, N_CB, T]."""
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()


def decode_tokens(codes):
    """Decode Mimi tokens to numpy audio."""
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()


def get_codebook_embeddings(cb_idx=0):
    """Extract codebook embedding matrix.
    Mimi uses MimiSplitResidualVectorQuantizer which splits into:
      - semantic_residual_vector_quantizer (CB0)
      - acoustic_residual_vector_quantizer (CB1-7)
    Each sub-quantizer has .layers[i].codebook.embed
    """
    q = mimi.quantizer
    # MimiSplitResidualVectorQuantizer path
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    # Fallback: flat .layers
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer type: {type(q).__name__}')

    # Get embedding from the layer's codebook
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3:
                w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE:
                return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')


cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embedding: {cb0_emb.shape}')

## 5. Dataset Preparation

Using **LibriSpeech test-clean**: ~40 speakers, ~5.4 hours total.

We select `N_SOURCE_SPEAKERS` + `N_TARGET_SPEAKERS` speakers with the most data.
For each target speaker, 80% of utterances are used for index/training, 20% held out.

In [ ]:
# ============================================================
# 5. DOWNLOAD & PREPARE DATASET
# ============================================================

print('Downloading LibriSpeech test-clean...')
ds = load_dataset(
    'hf-internal-testing/librispeech_asr_dummy', 'clean',
    split='validation', trust_remote_code=True
)
# For full experiment, replace with:
# ds = load_dataset('librispeech_asr', 'clean', split='test')

# Group by speaker, resample to Mimi SR
speaker_utts = defaultdict(list)  # sid -> [(wav_np, text), ...]
for sample in ds:
    sid = sample['speaker_id']
    wav = torch.tensor(sample['audio']['array'], dtype=torch.float32)
    orig_sr = sample['audio']['sampling_rate']
    if orig_sr != SR:
        wav = torchaudio.functional.resample(wav, orig_sr, SR)
    text = sample.get('text', '')
    speaker_utts[sid].append((wav.numpy(), text))

# Sort speakers by total audio duration (descending)
speaker_durations = {
    sid: sum(len(w) for w, _ in utts) / SR
    for sid, utts in speaker_utts.items()
}
sorted_speakers = sorted(speaker_durations, key=speaker_durations.get, reverse=True)

print(f'\nTotal speakers: {len(sorted_speakers)}')
for sid in sorted_speakers:
    print(f'  Speaker {sid}: {len(speaker_utts[sid])} utts, '
          f'{speaker_durations[sid]:.1f}s')

# Select source and target speakers (non-overlapping)
n_available = len(sorted_speakers)
if n_available >= N_SOURCE_SPEAKERS + N_TARGET_SPEAKERS:
    source_speakers = sorted_speakers[:N_SOURCE_SPEAKERS]
    target_speakers = sorted_speakers[N_SOURCE_SPEAKERS:N_SOURCE_SPEAKERS + N_TARGET_SPEAKERS]
elif n_available >= 2:
    # Few speakers: split roughly in half ensuring no overlap
    n_src = max(1, n_available // 2)
    source_speakers = sorted_speakers[:n_src]
    target_speakers = sorted_speakers[n_src:]
else:
    # Only 1 speaker: use for both (same-speaker test)
    source_speakers = sorted_speakers[:1]
    target_speakers = sorted_speakers[:1]

print(f'\nSource speakers: {source_speakers}')
print(f'Target speakers: {target_speakers}')

# Split target speaker data: 80% train, 20% eval
target_train = {}  # sid -> [(wav, text), ...]
target_eval = {}   # sid -> [(wav, text), ...]
for sid in target_speakers:
    utts = speaker_utts[sid]
    n_train = max(1, int(len(utts) * 0.8))
    target_train[sid] = utts[:n_train]
    target_eval[sid] = utts[n_train:] if n_train < len(utts) else utts[-1:]
    print(f'  Target {sid}: {n_train} train, {len(target_eval[sid])} eval')

# Source evaluation utterances
source_eval = {}
for sid in source_speakers:
    source_eval[sid] = speaker_utts[sid][:N_EVAL_UTTS]
    print(f'  Source {sid}: {len(source_eval[sid])} eval utts')

In [ ]:
# ============================================================
# 6. ENCODE ALL UTTERANCES WITH MIMI
# ============================================================

print('Encoding all utterances...')

# Encode source eval utterances
source_codes = {}  # (src_sid, utt_idx) -> codes [1, N_CB, T]
source_wavs = {}   # (src_sid, utt_idx) -> wav_np
source_texts = {}  # (src_sid, utt_idx) -> text

for sid in source_speakers:
    for i, (wav, text) in enumerate(source_eval[sid]):
        key = (sid, i)
        source_codes[key] = encode_audio(wav)
        source_wavs[key] = wav
        source_texts[key] = text

print(f'  Encoded {len(source_codes)} source utterances')

# Encode target train utterances
target_train_codes = {}  # tgt_sid -> list of codes
for sid in target_speakers:
    codes_list = []
    for wav, text in target_train[sid]:
        codes_list.append(encode_audio(wav))
    target_train_codes[sid] = codes_list
    total_frames = sum(c.shape[2] for c in codes_list)
    print(f'  Target {sid}: {len(codes_list)} utts, {total_frames} frames '
          f'({total_frames/FRAME_RATE:.1f}s)')

# Encode target eval utterances (for reference embeddings)
target_eval_wavs = {}
for sid in target_speakers:
    target_eval_wavs[sid] = [wav for wav, _ in target_eval[sid]]

print('Encoding complete.')

## 7. Voice Conversion Methods

In [ ]:
# ============================================================
# 7. IMPLEMENT ALL VC METHODS
# ============================================================

# --- Build per-target-speaker retrieval indices ---
target_indices = {}  # tgt_sid -> dict with 'faiss', 'lookup', etc.

for tgt_sid in target_speakers:
    codes_list = target_train_codes[tgt_sid]

    # Collect all frames
    all_sem = []
    all_acou = []
    all_full = []
    for codes in codes_list:
        T = codes.shape[2]
        for t_idx in range(T):
            all_sem.append(codes[0, 0, t_idx].item())
            all_acou.append(codes[0, 1:, t_idx].numpy())
            all_full.append(codes[0, :, t_idx].numpy())

    sem_arr = np.array(all_sem)
    acou_arr = np.stack(all_acou)
    full_arr = np.stack(all_full)

    # Lookup table
    lookup = defaultdict(list)
    for i, s in enumerate(all_sem):
        lookup[s].append(i)

    # FAISS index
    tgt_emb = cb0_emb[sem_arr].astype(np.float32)
    faiss.normalize_L2(tgt_emb)
    idx = faiss.IndexFlatIP(cb0_emb.shape[1])
    idx.add(tgt_emb)

    target_indices[tgt_sid] = {
        'lookup': lookup,
        'faiss': idx,
        'sem': sem_arr,
        'acou': acou_arr,
        'full': full_arr,
    }
    print(f'Target {tgt_sid}: {len(all_sem)} indexed frames, '
          f'{len(lookup)} unique semantic tokens')


# ---- Method 1: Naive Swap ----
def naive_swap(src_codes, tgt_sid):
    """Replace CB1-7 from a random target utterance (no alignment)."""
    tgt_codes = target_train_codes[tgt_sid][0]  # first target utt
    T = min(src_codes.shape[2], tgt_codes.shape[2])
    out = src_codes[:, :, :T].clone()
    out[:, 1:, :] = tgt_codes[:, 1:, :T]
    return out


# ---- Method 2: Retrieval Lookup ----
def retrieval_lookup(src_codes, tgt_sid):
    """Exact match on semantic tokens."""
    info = target_indices[tgt_sid]
    T = src_codes.shape[2]
    out = src_codes.clone()
    for t in range(T):
        sem = src_codes[0, 0, t].item()
        if sem in info['lookup']:
            idx = np.random.choice(info['lookup'][sem])
            out[0, :, t] = torch.tensor(info['full'][idx])
    return out


# ---- Method 3: Retrieval FAISS ----
def retrieval_faiss(src_codes, tgt_sid, k=4):
    """Soft match via FAISS cosine similarity."""
    info = target_indices[tgt_sid]
    T = src_codes.shape[2]
    src_sem = src_codes[0, 0].numpy()
    src_emb = cb0_emb[src_sem].astype(np.float32)
    faiss.normalize_L2(src_emb)
    dists, indices = info['faiss'].search(src_emb, k)
    out = src_codes.clone()
    for t in range(T):
        weights = F.softmax(torch.tensor(dists[t]) * 10, dim=0).numpy()
        chosen = np.random.choice(k, p=weights)
        out[0, :, t] = torch.tensor(info['full'][indices[t, chosen]])
    return out


# ---- Method 4: Retrieval FAISS + Smoothing ----
def retrieval_faiss_smooth(src_codes, tgt_sid, k=8, alpha=0.5):
    """FAISS retrieval with temporal smoothing."""
    info = target_indices[tgt_sid]
    T = src_codes.shape[2]
    src_sem = src_codes[0, 0].numpy()
    src_emb = cb0_emb[src_sem].astype(np.float32)
    faiss.normalize_L2(src_emb)
    dists, indices = info['faiss'].search(src_emb, k)

    selected = np.zeros(T, dtype=np.int64)
    selected[0] = indices[0, 0]
    for t in range(1, T):
        cands = indices[t]
        sem_scores = dists[t]
        temp_scores = 1.0 / (1.0 + np.abs(cands - selected[t-1]).astype(np.float32))
        combined = alpha * sem_scores + (1 - alpha) * temp_scores
        selected[t] = cands[np.argmax(combined)]

    out = src_codes.clone()
    for t in range(T):
        out[0, :, t] = torch.tensor(info['full'][selected[t]])
    return out


# ---- Method 5: Transformer ----
class AcousticTranslator(nn.Module):
    def __init__(self, n_sem=2048, n_acou=2048, n_cb=7,
                 d_model=256, n_heads=4, n_layers=4, max_len=1024):
        super().__init__()
        self.n_cb = n_cb
        self.sem_embed = nn.Embedding(n_sem, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout=0.1,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.heads = nn.ModuleList(
            [nn.Linear(d_model, n_acou) for _ in range(n_cb)]
        )

    def forward(self, sem_tokens):
        B, T = sem_tokens.shape
        x = self.sem_embed(sem_tokens) + self.pos_embed[:, :T]
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return [h(x) for h in self.heads]


def train_translator(tgt_sid, n_steps=TRANSFORMER_STEPS):
    """Train acoustic translator for a target speaker."""
    codes_list = target_train_codes[tgt_sid]
    all_codes = torch.cat(codes_list, dim=2)  # [1, 8, T_total]
    sem = all_codes[0, 0]       # [T]
    acou = all_codes[0, 1:]     # [7, T]
    T_total = sem.shape[0]

    model = AcousticTranslator(
        n_sem=CB_SIZE, n_acou=CB_SIZE, n_cb=N_CB-1
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps)

    model.train()
    for step in range(n_steps):
        starts = torch.randint(0, max(1, T_total - SEQ_LEN), (TRANSFORMER_BATCH,))
        sem_b = torch.stack([sem[s:s+SEQ_LEN] for s in starts]).to(device)
        acou_b = torch.stack([acou[:, s:s+SEQ_LEN] for s in starts]).to(device)

        logits = model(sem_b)
        loss = sum(
            F.cross_entropy(lg.reshape(-1, CB_SIZE), acou_b[:, i].reshape(-1))
            for i, lg in enumerate(logits)
        ) / len(logits)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

    model.eval()
    return model


@torch.no_grad()
def transformer_vc(src_codes, model, temperature=0.5, top_k=50):
    """Convert using trained translator."""
    sem = src_codes[0, 0:1].to(device)
    logits_list = model(sem)
    out = src_codes.clone()
    for i, logits in enumerate(logits_list):
        logits = logits / temperature
        if top_k > 0:
            topk_v, _ = torch.topk(logits, top_k, dim=-1)
            logits[logits < topk_v[:, :, -1:]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        pred = torch.multinomial(probs.reshape(-1, CB_SIZE), 1).reshape(1, -1)
        out[0, i + 1] = pred.cpu()
    return out


print('All methods defined.')

## 8. Evaluation Metrics

In [ ]:
# ============================================================
# 8. EVALUATION METRICS
# ============================================================

def compute_secs(wav_a, wav_b, sr=SR):
    """Speaker Embedding Cosine Similarity using Resemblyzer."""
    # Resemblyzer expects 16kHz
    wav_a_16 = librosa.resample(wav_a.astype(np.float32), orig_sr=sr, target_sr=16000)
    wav_b_16 = librosa.resample(wav_b.astype(np.float32), orig_sr=sr, target_sr=16000)
    emb_a = spk_encoder.embed_utterance(wav_a_16)
    emb_b = spk_encoder.embed_utterance(wav_b_16)
    cos = np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))
    return float(cos)


def compute_asr_cer(wav, reference_text, sr=SR):
    """Character Error Rate using Whisper ASR."""
    if not reference_text or not reference_text.strip():
        return None
    # Whisper expects float32, will resample internally
    result = whisper_model.transcribe(
        wav.astype(np.float32),
        language='en',
        fp16=(device == 'cuda')
    )
    hyp = result['text'].strip().upper()
    ref = reference_text.strip().upper()
    if not ref:
        return None
    return compute_cer(ref, hyp)


def compute_mcd(ref_wav, syn_wav, sr=SR, n_mfcc=13):
    """Mel Cepstral Distortion."""
    ref_mfcc = librosa.feature.mfcc(y=ref_wav.astype(np.float32), sr=sr, n_mfcc=n_mfcc)
    syn_mfcc = librosa.feature.mfcc(y=syn_wav.astype(np.float32), sr=sr, n_mfcc=n_mfcc)
    min_len = min(ref_mfcc.shape[1], syn_mfcc.shape[1])
    if min_len == 0:
        return None
    diff = ref_mfcc[1:, :min_len] - syn_mfcc[1:, :min_len]  # skip C0
    mcd = np.mean(np.sqrt(2.0 * np.sum(diff**2, axis=0)))
    return float(mcd)


def compute_pesq_score(ref_wav, deg_wav, sr=SR):
    """PESQ score (requires 16kHz)."""
    try:
        ref_16 = librosa.resample(ref_wav.astype(np.float32), orig_sr=sr, target_sr=16000)
        deg_16 = librosa.resample(deg_wav.astype(np.float32), orig_sr=sr, target_sr=16000)
        min_len = min(len(ref_16), len(deg_16))
        if min_len < 16000 * 0.5:  # at least 0.5s
            return None
        score = pesq_fn(16000, ref_16[:min_len], deg_16[:min_len], 'wb')
        return float(score)
    except Exception:
        return None


def evaluate_conversion(src_wav, conv_wav, tgt_wavs, src_text, sr=SR):
    """Compute all metrics for a single conversion."""
    results = {}

    # SECS: similarity between converted and target speaker
    secs_scores = []
    for tw in tgt_wavs:
        s = compute_secs(conv_wav, tw, sr)
        secs_scores.append(s)
    results['SECS'] = np.mean(secs_scores)

    # CER: content preservation
    cer_val = compute_asr_cer(conv_wav, src_text, sr)
    results['CER'] = cer_val

    # MCD: spectral distortion (vs source - lower means more similar to source)
    mcd_val = compute_mcd(src_wav, conv_wav, sr)
    results['MCD'] = mcd_val

    # PESQ: speech quality (vs source reconstruction)
    pesq_val = compute_pesq_score(src_wav, conv_wav, sr)
    results['PESQ'] = pesq_val

    return results


print('Evaluation functions defined.')

## 9. Run All Experiments

In [ ]:
# ============================================================
# 9. TRAIN TRANSFORMERS FOR EACH TARGET SPEAKER
# ============================================================

translators = {}  # tgt_sid -> trained model
train_times = {}  # tgt_sid -> seconds

for tgt_sid in target_speakers:
    print(f'Training translator for speaker {tgt_sid}...')
    t0 = time.time()
    translators[tgt_sid] = train_translator(tgt_sid)
    train_times[tgt_sid] = time.time() - t0
    n_params = sum(p.numel() for p in translators[tgt_sid].parameters())
    print(f'  Done in {train_times[tgt_sid]:.1f}s ({n_params/1e6:.1f}M params)')

print('\nAll translators trained.')

In [ ]:
# ============================================================
# 10. RUN ALL CONVERSIONS AND EVALUATE
# ============================================================

METHODS = [
    'naive_swap',
    'retrieval_lookup',
    'retrieval_faiss',
    'retrieval_faiss_smooth',
    'transformer',
]

all_results = []  # list of dicts with method, src, tgt, metrics
method_times = defaultdict(list)  # method -> list of RTF values

# Check if we have any cross-speaker pairs
has_cross_pairs = any(s != t for s in source_speakers for t in target_speakers)

total_pairs = len(source_speakers) * len(target_speakers)
pair_idx = 0

for src_sid in source_speakers:
    for tgt_sid in target_speakers:
        # Skip same-speaker only if we have cross-speaker pairs available
        if src_sid == tgt_sid and has_cross_pairs:
            continue
        pair_idx += 1
        print(f'\n[{pair_idx}/{total_pairs}] {src_sid} -> {tgt_sid}')

        tgt_ref_wavs = target_eval_wavs.get(tgt_sid, [w for w, _ in target_train[tgt_sid][:3]])

        for utt_i in range(len(source_eval.get(src_sid, []))):
            key = (src_sid, utt_i)
            if key not in source_codes:
                continue
            src_c = source_codes[key]
            src_w = source_wavs[key]
            src_t = source_texts[key]
            audio_dur = len(src_w) / SR

            for method in METHODS:
                np.random.seed(42 + utt_i)
                torch.manual_seed(42 + utt_i)

                # --- Convert ---
                t0 = time.time()
                if method == 'naive_swap':
                    conv_codes = naive_swap(src_c, tgt_sid)
                elif method == 'retrieval_lookup':
                    conv_codes = retrieval_lookup(src_c, tgt_sid)
                elif method == 'retrieval_faiss':
                    conv_codes = retrieval_faiss(src_c, tgt_sid)
                elif method == 'retrieval_faiss_smooth':
                    conv_codes = retrieval_faiss_smooth(src_c, tgt_sid)
                elif method == 'transformer':
                    conv_codes = transformer_vc(src_c, translators[tgt_sid])

                conv_wav = decode_tokens(conv_codes)
                elapsed = time.time() - t0
                rtf = elapsed / audio_dur if audio_dur > 0 else 0
                method_times[method].append(rtf)

                # --- Evaluate ---
                metrics = evaluate_conversion(
                    src_w, conv_wav, tgt_ref_wavs, src_t
                )
                metrics['RTF'] = rtf
                metrics['method'] = method
                metrics['src_speaker'] = src_sid
                metrics['tgt_speaker'] = tgt_sid
                metrics['utt_idx'] = utt_i
                all_results.append(metrics)

            print(f'  utt {utt_i}: done ({audio_dur:.1f}s)')

print(f'\nTotal evaluations: {len(all_results)}')
if not all_results:
    print('WARNING: No conversions were run. Check speaker selection above.')

## 10. Results — Table I (Objective Evaluation)

In [ ]:
# ============================================================
# 11. GENERATE TABLE I: OBJECTIVE RESULTS
# ============================================================

df = pd.DataFrame(all_results)

# Aggregate by method
method_labels = {
    'naive_swap': 'Naive Swap',
    'retrieval_lookup': 'Mimi-VC (Lookup)',
    'retrieval_faiss': 'Mimi-VC (FAISS)',
    'retrieval_faiss_smooth': 'Mimi-VC (FAISS+Smooth)',
    'transformer': 'Mimi-VC (Transformer)',
}

table_rows = []
for method in METHODS:
    subset = df[df['method'] == method]
    row = {
        'Method': method_labels[method],
        'SECS (mean)': f"{subset['SECS'].mean():.3f}",
        'SECS (std)': f"{subset['SECS'].std():.3f}",
        'CER% (mean)': f"{subset['CER'].dropna().mean()*100:.1f}",
        'MCD (mean)': f"{subset['MCD'].dropna().mean():.2f}",
        'PESQ (mean)': f"{subset['PESQ'].dropna().mean():.2f}",
        'RTF (mean)': f"{subset['RTF'].mean():.4f}",
    }
    table_rows.append(row)

table_I = pd.DataFrame(table_rows)
print('\n' + '='*80)
print('TABLE I: Objective Evaluation Results')
print('='*80)
print(table_I.to_string(index=False))
print()

# Save
table_I.to_csv(f'{RESULTS_DIR}/table_I_objective.csv', index=False)
print(f'Saved to {RESULTS_DIR}/table_I_objective.csv')

In [ ]:
# ============================================================
# 12. TABLE III: COMPUTATIONAL COST
# ============================================================

comp_rows = []
for method in METHODS:
    rtf_vals = method_times[method]
    label = method_labels[method]
    params = '0' if 'retrieval' in method or 'naive' in method else '~5M'
    if method == 'naive_swap':
        params = '0'
    comp_rows.append({
        'Method': label,
        'Params': params,
        'Training': 'No' if 'retrieval' in method or 'naive' in method else f'{np.mean(list(train_times.values())):.0f}s',
        f'RTF ({device.upper()})': f'{np.mean(rtf_vals):.4f}',
    })

table_III = pd.DataFrame(comp_rows)
print('\n' + '='*80)
print('TABLE III: Computational Cost')
print('='*80)
print(table_III.to_string(index=False))

table_III.to_csv(f'{RESULTS_DIR}/table_III_cost.csv', index=False)
print(f'\nSaved to {RESULTS_DIR}/table_III_cost.csv')

## 11. Ablation Study — Table IV (Codebook Analysis)

In [ ]:
# ============================================================
# 13. ABLATION: CODEBOOK ROLE ANALYSIS (TABLE IV)
# ============================================================
# For each configuration, swap specific codebooks between source
# and target, then measure SECS and CER.

ablation_configs = [
    ('None (source)', []),
    ('CB0 only', [0]),
    ('CB1 only', [1]),
    ('CB1-3', [1, 2, 3]),
    ('CB1-5', [1, 2, 3, 4, 5]),
    ('CB1-7 (all acoustic)', [1, 2, 3, 4, 5, 6, 7]),
    ('All (CB0-7)', [0, 1, 2, 3, 4, 5, 6, 7]),
]

abl_results = []

# Use first source-target pair for ablation
abl_src = source_speakers[0]
abl_tgt = target_speakers[0] if target_speakers[0] != abl_src else target_speakers[-1]

# Get smoothed target codes for consistent ablation
abl_src_key = (abl_src, 0)
abl_src_codes = source_codes[abl_src_key]
abl_src_wav = source_wavs[abl_src_key]
abl_src_text = source_texts[abl_src_key]
np.random.seed(42)
abl_tgt_codes = retrieval_faiss_smooth(abl_src_codes, abl_tgt)
abl_tgt_ref = target_eval_wavs.get(abl_tgt, [w for w, _ in target_train[abl_tgt][:3]])

print(f'Ablation: {abl_src} -> {abl_tgt}')
print(f'Source: {len(abl_src_wav)/SR:.1f}s\n')

for config_name, swap_cbs in ablation_configs:
    T = min(abl_src_codes.shape[2], abl_tgt_codes.shape[2])
    abl_codes = abl_src_codes[:, :, :T].clone()
    for cb in swap_cbs:
        abl_codes[0, cb, :] = abl_tgt_codes[0, cb, :T]

    abl_wav = decode_tokens(abl_codes)
    secs = compute_secs(abl_wav, abl_tgt_ref[0])
    cer_val = compute_asr_cer(abl_wav, abl_src_text)

    abl_results.append({
        'Swapped CBs': config_name,
        'SECS': f'{secs:.3f}',
        'CER%': f'{cer_val*100:.1f}' if cer_val is not None else 'N/A',
    })
    print(f'  {config_name}: SECS={secs:.3f}, CER={cer_val*100:.1f}%' if cer_val else
          f'  {config_name}: SECS={secs:.3f}, CER=N/A')

table_IV = pd.DataFrame(abl_results)
print('\n' + '='*80)
print('TABLE IV: Codebook Ablation Study')
print('='*80)
print(table_IV.to_string(index=False))

table_IV.to_csv(f'{RESULTS_DIR}/table_IV_ablation.csv', index=False)

In [ ]:
# ============================================================
# 14. ABLATION: RETRIEVAL PARAMETERS
# ============================================================
# Sweep k (number of neighbors) and alpha (smoothing weight)

k_values = [1, 2, 4, 8, 16, 32]
alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]

# Use same source/target pair
ret_abl = []

print('Retrieval parameter sweep...')

# Sweep k (with fixed alpha=0.5)
for k in k_values:
    np.random.seed(42)
    conv_codes = retrieval_faiss_smooth(abl_src_codes, abl_tgt, k=k, alpha=0.5)
    conv_wav = decode_tokens(conv_codes)
    secs = compute_secs(conv_wav, abl_tgt_ref[0])
    cer_val = compute_asr_cer(conv_wav, abl_src_text)
    ret_abl.append({'param': 'k', 'value': k, 'alpha': 0.5,
                    'SECS': secs,
                    'CER': cer_val if cer_val else 0})
    print(f'  k={k}: SECS={secs:.3f}')

# Sweep alpha (with fixed k=8)
for alpha in alpha_values:
    np.random.seed(42)
    conv_codes = retrieval_faiss_smooth(abl_src_codes, abl_tgt, k=8, alpha=alpha)
    conv_wav = decode_tokens(conv_codes)
    secs = compute_secs(conv_wav, abl_tgt_ref[0])
    cer_val = compute_asr_cer(conv_wav, abl_src_text)
    ret_abl.append({'param': 'alpha', 'value': alpha, 'k': 8,
                    'SECS': secs,
                    'CER': cer_val if cer_val else 0})
    print(f'  alpha={alpha}: SECS={secs:.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

k_data = [r for r in ret_abl if r['param'] == 'k']
axes[0].plot([r['value'] for r in k_data], [r['SECS'] for r in k_data], 'o-', color='#4472C4')
axes[0].set_xlabel('k (number of neighbors)')
axes[0].set_ylabel('SECS')
axes[0].set_title('Effect of k on Speaker Similarity')
axes[0].grid(True, alpha=0.3)

a_data = [r for r in ret_abl if r['param'] == 'alpha']
axes[1].plot([r['value'] for r in a_data], [r['SECS'] for r in a_data], 's-', color='#C55A11')
axes[1].set_xlabel('alpha (smoothing weight)')
axes[1].set_ylabel('SECS')
axes[1].set_title('Effect of alpha on Speaker Similarity')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/retrieval_ablation.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/retrieval_ablation.png')

In [ ]:
# ============================================================
# 15. ABLATION: TRANSFORMER DESIGN
# ============================================================
# Sweep: temperature, number of layers, training steps

trans_abl = []
base_model = translators[abl_tgt]

# --- Temperature sweep ---
print('Temperature sweep...')
for temp in [0.1, 0.2, 0.5, 0.8, 1.0, 1.5]:
    torch.manual_seed(42)
    conv_codes = transformer_vc(abl_src_codes, base_model, temperature=temp)
    conv_wav = decode_tokens(conv_codes)
    secs = compute_secs(conv_wav, abl_tgt_ref[0])
    cer_val = compute_asr_cer(conv_wav, abl_src_text)
    trans_abl.append({'param': 'temp', 'value': temp,
                      'SECS': secs, 'CER': cer_val if cer_val else 0})
    print(f'  temp={temp}: SECS={secs:.3f}, CER={cer_val*100:.1f}%' if cer_val else
          f'  temp={temp}: SECS={secs:.3f}')

# --- Training steps sweep ---
print('\nTraining steps sweep...')
for n_steps in [50, 100, 250, 500, 1000]:
    torch.manual_seed(42)
    tmp_model = train_translator(abl_tgt, n_steps=n_steps)
    conv_codes = transformer_vc(abl_src_codes, tmp_model, temperature=0.5)
    conv_wav = decode_tokens(conv_codes)
    secs = compute_secs(conv_wav, abl_tgt_ref[0])
    trans_abl.append({'param': 'steps', 'value': n_steps, 'SECS': secs})
    print(f'  steps={n_steps}: SECS={secs:.3f}')
    del tmp_model

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

temp_data = [r for r in trans_abl if r['param'] == 'temp']
axes[0].plot([r['value'] for r in temp_data], [r['SECS'] for r in temp_data], 'o-', color='#7030A0')
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('SECS')
axes[0].set_title('Effect of Sampling Temperature')
axes[0].grid(True, alpha=0.3)

step_data = [r for r in trans_abl if r['param'] == 'steps']
axes[1].plot([r['value'] for r in step_data], [r['SECS'] for r in step_data], 's-', color='#548235')
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('SECS')
axes[1].set_title('Effect of Training Duration')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/transformer_ablation.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/transformer_ablation.png')

## 12. Visualizations

In [ ]:
# ============================================================
# 16. TOKEN VISUALIZATION (FIGURE 2)
# ============================================================

np.random.seed(42)
torch.manual_seed(42)
vis_src = abl_src_codes
vis_ret = retrieval_faiss_smooth(vis_src, abl_tgt)
vis_trans = transformer_vc(vis_src, translators[abl_tgt])
vis_tgt = target_train_codes[abl_tgt][0]

L = min(200, vis_src.shape[2], vis_tgt.shape[2])

fig, axes = plt.subplots(4, 1, figsize=(14, 10), constrained_layout=True)
titles = [
    f'Source Speaker {abl_src}',
    'Retrieval VC (FAISS+Smooth)',
    'Transformer VC (t=0.5)',
    f'Target Speaker {abl_tgt} (reference)',
]
code_list = [
    vis_src[0, :, :L],
    vis_ret[0, :, :L],
    vis_trans[0, :, :L],
    vis_tgt[0, :, :L],
]

for ax, title, codes in zip(axes, titles, code_list):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('Codebook')
    ax.set_title(title, fontsize=11)
    ax.set_yticks(range(N_CB))
    ax.set_yticklabels([f'CB{i}' for i in range(N_CB)])
    plt.colorbar(im, ax=ax, shrink=0.6)

axes[-1].set_xlabel('Frame (12.5 Hz)')
plt.suptitle('Figure 2: Mimi Token Comparison Across Methods', fontsize=13)
plt.savefig(f'{RESULTS_DIR}/figure2_tokens.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/figure2_tokens.png')

In [ ]:
# ============================================================
# 17. BAR CHART: METHOD COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

metrics_to_plot = ['SECS', 'CER', 'MCD', 'PESQ']
colors = ['#4472C4', '#C55A11', '#548235', '#7030A0', '#BF8F00']

for ax, metric in zip(axes, metrics_to_plot):
    vals = []
    errs = []
    labels = []
    for i, method in enumerate(METHODS):
        subset = df[df['method'] == method][metric].dropna()
        if metric == 'CER':
            subset = subset * 100  # percentage
        vals.append(subset.mean())
        errs.append(subset.std())
        labels.append(method_labels[method].replace('Mimi-VC ', ''))

    bars = ax.bar(range(len(vals)), vals, yerr=errs, color=colors,
                  capsize=3, edgecolor='#333333', linewidth=0.5)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    unit = '%' if metric == 'CER' else ''
    ax.set_ylabel(f'{metric}{unit}')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Objective Metrics Comparison', fontsize=13)
plt.savefig(f'{RESULTS_DIR}/metrics_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 13. Export Audio for Subjective Evaluation (MOS/SMOS)

In [ ]:
# ============================================================
# 18. EXPORT AUDIO SAMPLES FOR LISTENING TEST
# ============================================================
# Generate audio files for MOS/SMOS evaluation

import soundfile as sf

AUDIO_DIR = f'{RESULTS_DIR}/audio_samples'
os.makedirs(AUDIO_DIR, exist_ok=True)

# Export for first 3 source-target pairs
n_export = 0
for src_sid in source_speakers[:2]:
    for tgt_sid in target_speakers[:2]:
        if src_sid == tgt_sid:
            continue
        for utt_i in range(min(3, len(source_eval[src_sid]))):
            key = (src_sid, utt_i)
            if key not in source_codes:
                continue

            prefix = f'{AUDIO_DIR}/s{src_sid}_t{tgt_sid}_u{utt_i}'

            # Source original
            sf.write(f'{prefix}_source.wav', source_wavs[key], SR)

            # Target reference
            tgt_ref = target_eval_wavs.get(tgt_sid, [w for w, _ in target_train[tgt_sid][:1]])
            if tgt_ref:
                sf.write(f'{prefix}_target_ref.wav', tgt_ref[0], SR)

            # Converted samples
            np.random.seed(42)
            torch.manual_seed(42)
            src_c = source_codes[key]

            for method in ['retrieval_faiss_smooth', 'transformer']:
                if method == 'retrieval_faiss_smooth':
                    conv = retrieval_faiss_smooth(src_c, tgt_sid)
                else:
                    conv = transformer_vc(src_c, translators[tgt_sid])
                wav = decode_tokens(conv)
                sf.write(f'{prefix}_{method}.wav', wav, SR)

            n_export += 1

print(f'Exported {n_export} sample sets to {AUDIO_DIR}/')
print('Each set contains: source, target_ref, retrieval, transformer')
print('\nUse these for MOS/SMOS listening tests (Table II in paper).')

In [ ]:
# ============================================================
# 19. LISTEN TO SAMPLES IN NOTEBOOK
# ============================================================

# Pick one example to listen to
demo_src = source_speakers[0]
demo_tgt = target_speakers[0] if target_speakers[0] != demo_src else target_speakers[-1]
demo_key = (demo_src, 0)

if demo_key in source_codes:
    print(f'Demo: Speaker {demo_src} -> Speaker {demo_tgt}\n')

    display(PlayAudio(source_wavs[demo_key], rate=SR))
    print('>> Source (original)\n')

    for method in METHODS:
        np.random.seed(42)
        torch.manual_seed(42)
        sc = source_codes[demo_key]

        if method == 'naive_swap':
            cc = naive_swap(sc, demo_tgt)
        elif method == 'retrieval_lookup':
            cc = retrieval_lookup(sc, demo_tgt)
        elif method == 'retrieval_faiss':
            cc = retrieval_faiss(sc, demo_tgt)
        elif method == 'retrieval_faiss_smooth':
            cc = retrieval_faiss_smooth(sc, demo_tgt)
        elif method == 'transformer':
            cc = transformer_vc(sc, translators[demo_tgt])

        cw = decode_tokens(cc)
        display(PlayAudio(cw, rate=SR))
        print(f'>> {method_labels[method]}\n')

    tgt_ref = target_eval_wavs.get(demo_tgt, [w for w, _ in target_train[demo_tgt][:1]])
    if tgt_ref:
        display(PlayAudio(tgt_ref[0], rate=SR))
        print('>> Target speaker (reference)\n')

## 14. Final Summary & Export

In [ ]:
# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print('='*80)
print('EXPERIMENT SUMMARY')
print('='*80)
print(f'\nDataset: LibriSpeech test-clean')
print(f'Source speakers: {source_speakers}')
print(f'Target speakers: {target_speakers}')
print(f'Total conversions evaluated: {len(all_results)}')
print(f'\nDevice: {device}')
print(f'Mimi: {N_CB} codebooks x {CB_SIZE}, {FRAME_RATE} Hz')
print(f'Transformer: {TRANSFORMER_STEPS} steps, {TRANSFORMER_BATCH} batch')

print('\n' + '='*80)
print('TABLE I: Objective Evaluation')
print('='*80)
print(table_I.to_string(index=False))

print('\n' + '='*80)
print('TABLE III: Computational Cost')
print('='*80)
print(table_III.to_string(index=False))

print('\n' + '='*80)
print('TABLE IV: Codebook Ablation')
print('='*80)
print(table_IV.to_string(index=False))

# Save all raw results
df.to_csv(f'{RESULTS_DIR}/all_results_raw.csv', index=False)

# Save ablation results
pd.DataFrame(ret_abl).to_csv(f'{RESULTS_DIR}/retrieval_ablation.csv', index=False)
pd.DataFrame(trans_abl).to_csv(f'{RESULTS_DIR}/transformer_ablation.csv', index=False)

print(f'\nAll results saved to {RESULTS_DIR}/')
print('\nFiles:')
for f in sorted(os.listdir(RESULTS_DIR)):
    if not f.startswith('.'):
        fpath = os.path.join(RESULTS_DIR, f)
        if os.path.isfile(fpath):
            print(f'  {f}')

In [ ]:
# ============================================================
# 21. DOWNLOAD RESULTS (Colab)
# ============================================================
# Zip all results for easy download

!cd /content && zip -r mimi_vc_results.zip results/

try:
    from google.colab import files
    files.download('/content/mimi_vc_results.zip')
    print('Download started!')
except ImportError:
    print(f'Results saved to /content/mimi_vc_results.zip')
    print('Download manually or use: cp /content/mimi_vc_results.zip /your/path')